# CS-781 Capstone — Phase 1: LIGHTWEIGHT architecture — a 3-layer LSTM language model (~84M parameters), fast to train

The "lightweight / simple" architecture entry: **no attention at all**. A tied-embedding LSTM (embedding 768, 3 LSTM layers of 1536 units, projection back to 768, output layer tied to the embedding) trained from scratch on the same BPE tokenizer as the other models (the saved 160M-word one from the bucket, nothing is retrained).

| | |
|---|---|
| data | Supreme Court 70M + acts 30M words (random documents, held-out Supreme Court documents excluded). The huge high-court / tribunal files are NOT read, which keeps data preparation short |
| training | fp16 mixed precision, AdamW, cosine learning rate, **time-limited**: it trains for `TRAIN_MINUTES` (default 90) or one pass over the data, whichever comes first; the schedule follows whichever is further along. Uses both GPUs through `nn.DataParallel` when two are available |
| decoding | tries three settings (greedy + no-repeat-4, and sampling at T 0.3 / 0.7 with no-repeat-4) on held-out Supreme Court windows, and writes the submission with the best by BERTScore |
| resume | the checkpoint (model, optimizer, step, elapsed time) is saved every few minutes and pushed to the bucket; re-run from the top to continue after an interruption |

Set `QUICK_TEST = True` first (a couple of minutes), then `False`.

## 1. Settings

In [1]:
import time as _t0mod
NB_START = _t0mod.time()
QUICK_TEST = False          # True = a ~3-minute pipeline test

# ---- Data -------------------------------------------------------------------------------------------------------------------
PREVIOUS_WORDS_PER_SOURCE = 0            # nothing is skipped: this is a from-scratch model
EXTRA_PREVIOUS_STAGES = {}
if QUICK_TEST:
    NEW_WORD_QUOTAS = {"supreme_court": 300_000, "high_courts": 0, "acts": 200_000, "tribunals": 0}
else:
    NEW_WORD_QUOTAS = {"supreme_court": 20_000_000, "high_courts": 20_000_000, "acts": 20_000_000, "tribunals": 20_000_000}
MAX_WORDS_FOR_TRAINING = sum(NEW_WORD_QUOTAS.values())
NUM_EPOCHS = 1

# ---- Model ----------------------------------------------------------------------------------------------------------------
VOCAB_SIZE = 40000
EMBED_DIM = 768
LSTM_HIDDEN = 1536
LSTM_LAYERS = 3
CONTEXT_LEN = 512                         # training sequence length (the LSTM carries its state beyond this at generation time)
PROMPT_KEEP = 768                         # prompt tokens fed to the LSTM when generating

# ---- Training -------------------------------------------------------------------------------------------------------------
TRAIN_MINUTES = 3 if QUICK_TEST else 90   # time budget of the training loop
PER_GPU_BATCH = 4 if QUICK_TEST else 16   # sequences (of CONTEXT_LEN tokens) per GPU per step
PEAK_LR = 1.5e-3
WARMUP_STEPS = 20 if QUICK_TEST else 300
MIN_LR_FRACTION = 0.1
GRAD_CLIP = 1.0
WEIGHT_DECAY = 0.01
DATA_SEED = 2000
INIT_SEED = 0
CKPT_EVERY_MINUTES = 1 if QUICK_TEST else 12
BATCH_SIZE = 16                           # held-out loss evaluation batch
TEMPERATURE = 0.3
LEARNING_RATE = PEAK_LR

RUN_TAG = "lstm84M"
mode_tag = "quicktest" if QUICK_TEST else "full"
TOKENIZER_RUN_TAG = "160Mw_1ep"
TOKENIZER_WORD_COUNT = 160_000_000
REQUIRE_SAVED_TOKENIZER = True
NORM_TYPE = "layernorm"; NORM_POSITION = "pre"; OPTIMIZER_TYPE = "adamw"
CHECKPOINT_NAME = f"phase1_lightweight_lstm84M_{mode_tag}_checkpoint.pt"
CHECKPOINT_PATH = "/kaggle/working/" + CHECKPOINT_NAME


## 2. Make sure the working folder exists

In [2]:
import os
os.makedirs("/kaggle/working", exist_ok=True)
print("Working folder ready:", os.path.exists("/kaggle/working"))

Working folder ready: True


## 3. Imports and Hugging Face login

In [3]:
!pip install -q -U "huggingface_hub>=1.5"   # storage buckets need huggingface_hub >= 1.5
import json
import os
import zipfile
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 13.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.1.1 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.1.1 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.1.1 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.1.1 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.1.1 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.
Using device: cuda


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 3b. Model storage (the bucket) and run log

In [4]:
from huggingface_hub import HfApi, create_repo, create_bucket, batch_bucket_files, download_bucket_files, list_bucket_tree, hf_hub_download
import shutil

HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
hf_api = HfApi(token=HF_WRITE_TOKEN)

# ---- Where models live: a Hugging Face STORAGE BUCKET (not a repo) -------------------------------------------
# A repo is a git repository: every overwritten checkpoint stays in its history, so a few 3 GB models quickly
# cost tens of GB. A bucket is plain mutable storage: overwriting a file REPLACES it (no history), so each model
# costs exactly its own size. Needs huggingface_hub >= 1.5 (installed in Section 3).
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # the OLD repo: only READ, as a fallback for models not moved yet
create_bucket(HF_BUCKET, private=True, exist_ok=True, token=HF_WRITE_TOKEN)


def list_model_files():
    """Names of every file in the bucket (empty list if the bucket cannot be listed)."""
    try:
        return [f.path for f in list_bucket_tree(HF_BUCKET, recursive=True, token=HF_WRITE_TOKEN)
                if getattr(f, "type", "file") == "file"]
    except Exception as e:
        print(f"(could not list the bucket: {type(e).__name__})")
        return []


def upload_model_file(local_path, name=None, quiet=False):
    """Upload local_path to the bucket as `name` (default: its own filename). Replaces any older copy."""
    name = name or os.path.basename(local_path)
    batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
    if not quiet:
        print(f"Uploaded {name} to bucket {HF_BUCKET}")


def fetch_model_file(name, local_path=None):
    """Download `name` from the bucket (falling back to the old repo) to local_path; returns the local path.
    Raises FileNotFoundError if it is in neither place."""
    local_path = local_path or f"/kaggle/working/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        if not os.path.exists(part):
            raise FileNotFoundError(name)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    try:   # fallback: the old repo (read-only)
        p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
        shutil.copy(p, local_path)
        print(f"(found {name} only in the old repo {HF_CHECKPOINT_REPO}; new saves go to the bucket)")
        return local_path
    except Exception:
        raise FileNotFoundError(f"{name} is in neither the bucket {HF_BUCKET} nor the repo {HF_CHECKPOINT_REPO}")


def push_checkpoint_to_hub(local_path, quiet=False):
    """Upload a checkpoint, the tokenizer, or the run log to the bucket under its own filename (which already
    encodes the full config -- see Settings). Each upload REPLACES the previous version, and nothing is kept."""
    upload_model_file(local_path, quiet=quiet)


def pull_from_hub(local_path):
    """Quietly fetch local_path's file (by basename) if it is not on disk yet. True if it exists afterwards."""
    if os.path.exists(local_path):
        return True
    try:
        fetch_model_file(os.path.basename(local_path), local_path)
        print(f"Pulled {os.path.basename(local_path)} from the Hub")
        return True
    except Exception:
        return False


def try_pull_checkpoint_from_hub(local_path):
    """If no LOCAL checkpoint exists yet (fresh Kaggle session / other account), fetch a matching one."""
    if os.path.exists(local_path):
        return
    try:
        fetch_model_file(os.path.basename(local_path), local_path)
        print(f"Pulled matching checkpoint from the Hub: {os.path.basename(local_path)}")
    except Exception as e:
        print(f"No matching checkpoint on the Hub yet ({type(e).__name__}) -- starting fresh.")


print(f"Model storage configured -> bucket {HF_BUCKET}")

# ---- Run log: a running record of what this project did, for the final report ----
# Every important step appends one JSON line (time, what happened, the numbers) to
# RUN_LOG_PATH. The file is pushed to the Hub together with the checkpoints and pulled
# back at the start of a new session, so a run that spans several sessions (or both
# Kaggle accounts) keeps ONE continuous log. Read it with: print(open(RUN_LOG_PATH).read())
import time as _time, datetime as _dt
RUN_LOG_PATH = f"/kaggle/working/run_log_gqa_kvcache_{RUN_TAG}_{mode_tag}.jsonl"
pull_from_hub(RUN_LOG_PATH)


def log_event(kind, **fields):
    """Append one event to the run log (never raises: logging must not stop a training run)."""
    try:
        record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}
        with open(RUN_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, default=str) + "\n")
    except Exception as e:
        print(f"(run log write failed: {type(e).__name__})")


def sync_run_log():
    try:
        push_checkpoint_to_hub(RUN_LOG_PATH, quiet=True)
    except Exception as e:
        print(f"(run log push failed: {type(e).__name__})")


log_event("session_start", quick_test=QUICK_TEST, vocab_size=VOCAB_SIZE, context_len=CONTEXT_LEN,
          batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, num_epochs=NUM_EPOCHS,
          max_words=MAX_WORDS_FOR_TRAINING, norm=f"{NORM_TYPE}/{NORM_POSITION}", optimizer=OPTIMIZER_TYPE,
          device=device, gpu=(torch.cuda.get_device_name(0) if device == "cuda" else None))
print(f"Run log: {RUN_LOG_PATH}")

Model storage configured -> bucket kavincg23/cs781-models
Run log: /kaggle/working/run_log_gqa_kvcache_lstm84M_full.jsonl


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 4. Dataset names and the test prompts

In [5]:
!pip install -q ijson
import ijson

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 3.5 MB/s eta 0:00:00


In [6]:
TEXT_FIELD = "text"  # <-- change this if needed, based on what you saw above

In [7]:
test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE,
                                  repo_type="dataset", token=HF_TOKEN)
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_prompts = {str(doc_id): record[TEXT_FIELD] for doc_id, record in test_data.items()}

print(f"Loaded {len(test_prompts)} test prompts")
first_id = list(test_prompts.keys())[0]
print(f"Example prompt (id={first_id}), word count = {len(test_prompts[first_id].split())}")

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

Loaded 1000 test prompts
Example prompt (id=1), word count = 512


## 5. The training documents (and the held-out evaluation documents)

A seeded random order over each source file; the **held-out** Supreme Court documents come first (never trained on; the same ones as every other model), then the training documents until the quota of each source.

In [8]:
import array, random, time
SAMPLING_SEED = 42      # MUST match the base run: same seed -> same shuffle -> same held-out and previous documents
HELDOUT_WORDS = {'acts': 500000, 'high_courts': 500000, 'supreme_court': 2000000, 'tribunals': 500000}
HELDOUT_MIN_DOC_WORDS = 640
HELDOUT_MAX_WORDS_PER_DOC = 6000
HELDOUT_FILES = {}
CORPUS_FILE = "/kaggle/working/continue_training_corpus.txt"
print("Streaming through the training files...")

word_count = 0
new_words_by_source = {}
selection_debug = {}
with open(CORPUS_FILE, "w", encoding="utf-8") as out_f:
    for filename in TRAIN_FILES:
        short = os.path.basename(filename)[len("train_"):-len(".json")]
        new_quota = NEW_WORD_QUOTAS.get(short, 0)
        if new_quota <= 0 and short != "supreme_court":
            continue        # nothing to train on from this source (Supreme Court is always read: it supplies the held-out documents)
        local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)

        t0 = time.time()
        doc_lengths = array.array("l")
        with open(local_path, "r", encoding="utf-8") as f:
            for record in ijson.items(f, "item"):
                doc_lengths.append(len(record[TEXT_FIELD].split()))
        print(f"  {filename}: {len(doc_lengths):,} documents, {sum(doc_lengths):,} words (counted in {time.time() - t0:.0f}s)")

        order = list(range(len(doc_lengths)))
        random.Random(SAMPLING_SEED).shuffle(order)

        # 1) held-out documents (identical to the base run)
        held_target = HELDOUT_WORDS.get(short, 0)
        heldout_ids, held_words = set(), 0
        for i in order:
            if held_words >= held_target:
                break
            if doc_lengths[i] >= HELDOUT_MIN_DOC_WORDS:
                heldout_ids.add(i)
                held_words += min(doc_lengths[i], HELDOUT_MAX_WORDS_PER_DOC)

        # 2) the documents the earlier runs trained on, replayed stage by stage with the same rule (identical selection):
        #    stage 1 = the base run (PREVIOUS_WORDS_PER_SOURCE), later stages = the continuations' quotas (EXTRA_PREVIOUS_STAGES)
        previous_ids = set()
        for stage_quota in [PREVIOUS_WORDS_PER_SOURCE] + list(EXTRA_PREVIOUS_STAGES.get(short, [])):
            planned = 0
            stage_ids = set()
            for i in order:
                if planned >= stage_quota:
                    break
                if i in heldout_ids or i in previous_ids:
                    continue
                keep = min(doc_lengths[i], stage_quota - planned)
                if keep > 0:
                    stage_ids.add(i)
                    planned += keep
            previous_ids |= stage_ids

        # 3) NEW documents: the next ones in the same order
        chosen, planned = {}, 0
        for i in order:
            if planned >= new_quota:
                break
            if i in heldout_ids or i in previous_ids:
                continue
            keep = min(doc_lengths[i], new_quota - planned)
            if keep > 0:
                chosen[i] = keep
                planned += keep
        available = sum(doc_lengths) - sum(doc_lengths[i] for i in heldout_ids | previous_ids)
        if planned < new_quota:
            print(f"  NOTE: {short} only has {planned:,} unused words (asked for {new_quota:,}); using all of them.")
        del order
        selection_debug[short] = (heldout_ids, previous_ids, set(chosen))     # (kept so the selection can be inspected)

        write_held = heldout_ids if short == "supreme_court" else set()     # only Supreme Court held-out text is used for evaluation
        source_word_count = 0
        remaining_docs, remaining_held = len(chosen), len(write_held)
        held_f = None
        if write_held:
            heldout_path = os.path.join(os.path.dirname(CORPUS_FILE), f"heldout_{short}.txt")
            held_f = open(heldout_path, "w", encoding="utf-8")
            HELDOUT_FILES[short] = heldout_path
        with open(local_path, "r", encoding="utf-8") as f:
            for i, record in enumerate(ijson.items(f, "item")):
                keep = chosen.get(i)
                if keep is None and i not in write_held:
                    continue
                doc_words = record[TEXT_FIELD].split()
                if keep is not None:
                    out_f.write(" ".join(doc_words[:keep]) + " ")
                    source_word_count += min(keep, len(doc_words))
                    remaining_docs -= 1
                else:
                    held_f.write(" ".join(doc_words[:HELDOUT_MAX_WORDS_PER_DOC]) + "\n")
                    remaining_held -= 1
                if remaining_docs == 0 and remaining_held == 0:
                    break
        if held_f is not None:
            held_f.close()
        word_count += source_word_count
        new_words_by_source[short] = source_word_count
        print(f"  ...{short}: {source_word_count:,} NEW words from {len(chosen):,} documents (quota {new_quota:,}, unused words in the file: {available:,}); "
              f"skipped {len(previous_ids):,} documents of the previous run; held out {len(heldout_ids):,} documents")

MAX_WORDS_FOR_TRAINING = word_count
print(f"\nNEW training words: {word_count:,}  (written to {CORPUS_FILE})   by source: {new_words_by_source}")
log_event("continue_data", new_words=word_count, by_source=new_words_by_source, previous_per_source=PREVIOUS_WORDS_PER_SOURCE, extra_previous_stages=EXTRA_PREVIOUS_STAGES)


Streaming through the training files...


Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

  Phase_1_Pretraining/Train/train_acts.json: 18,826 documents, 119,638,348 words (counted in 10s)
  ...acts: 20,000,000 NEW words from 3,128 documents (quota 20,000,000, unused words in the file: 118,684,097); skipped 0 documents of the previous run; held out 140 documents


Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

... (4 outputs left out)

Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

  Phase_1_Pretraining/Train/train_tribunals.json: 869,132 documents, 975,917,446 words (counted in 86s)
  ...tribunals: 20,000,000 NEW words from 18,026 documents (quota 20,000,000, unused words in the file: 975,346,913); skipped 0 documents of the previous run; held out 273 documents

NEW training words: 80,000,000  (written to /kaggle/working/continue_training_corpus.txt)   by source: {'acts': 20000000, 'high_courts': 20000000, 'supreme_court': 20000000, 'tribunals': 20000000}


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 6. The saved tokenizer

In [9]:
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)


# The tokenizer depends only on the training corpus, VOCAB_SIZE and BPE_MIN_WORD_FREQ, so its file name records
# exactly those (the corpus is fixed by the sampling seed, the quotas and the word count). If a file with this name is
# on disk or on the Hub it is loaded (skipping the slow pure-Python training); otherwise it is trained, saved and pushed.
TOKENIZER_PATH = (f"/kaggle/working/a1bpe_tokenizer_{TOKENIZER_RUN_TAG}_{mode_tag}_v{VOCAB_SIZE}_{TOKENIZER_WORD_COUNT}w"
                  f"_seed{SAMPLING_SEED}_minf{BPE_MIN_WORD_FREQ}.json")
t_start = time.time()
if pull_from_hub(TOKENIZER_PATH):
    print(f"Loading the saved tokenizer {os.path.basename(TOKENIZER_PATH)} (skipping training)...")
    bpe_tokenizer = BPETokenizer.load(TOKENIZER_PATH)
    tokenizer_source = "loaded"
else:
    if REQUIRE_SAVED_TOKENIZER:
        raise RuntimeError(f"The saved tokenizer {os.path.basename(TOKENIZER_PATH)} was not found locally or in the bucket {HF_BUCKET}. "
                           "This notebook only decodes with the trained model, so it needs the SAME tokenizer the model was trained with. "
                           "Check the HF token / bucket and the tag/word count in Settings.")
    print("Training the BPE tokenizer (our own implementation, pure Python)...")
    bpe_tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE).fit_file(CORPUS_FILE, min_word_freq=BPE_MIN_WORD_FREQ)
    bpe_tokenizer.save(TOKENIZER_PATH, meta={"word_count": word_count, "sampling_seed": SAMPLING_SEED,
                                             "min_word_freq": BPE_MIN_WORD_FREQ, "requested_vocab": VOCAB_SIZE})
    try:
        push_checkpoint_to_hub(TOKENIZER_PATH)
    except Exception as e:
        print(f"(could not push the tokenizer to the Hub: {type(e).__name__} -- it is still saved locally)")
    tokenizer_source = "trained"
print(f"Tokenizer {tokenizer_source} in {time.time() - t_start:.0f}s")
log_event("tokenizer", source=tokenizer_source, seconds=round(time.time() - t_start), vocab_size=bpe_tokenizer.get_vocab_size(),
          merges=len(bpe_tokenizer.merges), words=word_count, file=os.path.basename(TOKENIZER_PATH))


def encode(text):
    """Turn a string into a list of BPE token IDs."""
    return bpe_tokenizer.encode(text)


def decode(ids):
    """Turn a list of BPE token IDs back into a string."""
    return bpe_tokenizer.decode(ids)


print("Vocabulary size:", bpe_tokenizer.get_vocab_size())
if bpe_tokenizer.get_vocab_size() < VOCAB_SIZE:
    print(f"NOTE: fewer than VOCAB_SIZE={VOCAB_SIZE} merges were possible; the model uses the real size above.")

# Encode the NEW training corpus (streamed from disk). all_word_ids holds BPE-token ids.
t_start = time.time()
all_word_ids = bpe_tokenizer.encode_file(CORPUS_FILE)
print(f"{word_count:,} words became {len(all_word_ids):,} BPE tokens "
      f"({len(all_word_ids) / max(1, word_count):.2f} tokens per word) in {time.time() - t_start:.0f}s")
log_event("encoded_corpus", words=word_count, tokens=len(all_word_ids), tokens_per_word=round(len(all_word_ids) / max(1, word_count), 3),
          seconds=round(time.time() - t_start))


(…)full_v40000_160000000w_seed42_minf1.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

(found a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json only in the old repo kavincg23/cs781-checkpoints; new saves go to the bucket)
Pulled a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json from the Hub
Loading the saved tokenizer a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json (skipping training)...
Tokenizer loaded in 1s


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


Vocabulary size: 40000
80,000,000 words became 91,783,996 BPE tokens (1.15 tokens per word) in 88s


## 7. The LSTM model

In [10]:
import torch, torch.nn as nn, torch.nn.functional as F, math, time, os, random

class LSTMLM(nn.Module):
    """Tied-embedding LSTM language model. forward(x) -> logits; forward(x, y) -> mean cross-entropy (computed inside, so with
    DataParallel only a scalar per GPU travels back). step(x, state) -> (logits of the LAST position, new state): used for generation."""
    def __init__(self, vocab, embed, hidden, layers):
        super().__init__()
        self.vocab = vocab
        self.emb = nn.Embedding(vocab, embed, padding_idx=0)
        self.lstm = nn.LSTM(embed, hidden, num_layers=layers, batch_first=True)
        self.proj = nn.Linear(hidden, embed)
        self.bias = nn.Parameter(torch.zeros(vocab))
        nn.init.normal_(self.emb.weight, std=0.02)
        with torch.no_grad():
            self.emb.weight[0].zero_()
        for name, p in self.lstm.named_parameters():
            if "bias" in name:
                nn.init.zeros_(p)
                n = p.numel() // 4
                p.data[n:2 * n].fill_(1.0)                 # forget-gate bias (ih and hh biases both 1 -> effective 2)
            else:
                nn.init.uniform_(p, -1.0 / math.sqrt(hidden), 1.0 / math.sqrt(hidden))

    def _logits(self, h):
        return F.linear(self.proj(h), self.emb.weight, self.bias)

    def forward(self, x, y=None):
        self.lstm.flatten_parameters()
        h, _ = self.lstm(self.emb(x))
        logits = self._logits(h)
        if y is None:
            return logits
        return F.cross_entropy(logits.float().reshape(-1, self.vocab), y.reshape(-1))

    def step(self, x, state=None):
        self.lstm.flatten_parameters()
        h, state = self.lstm(self.emb(x), state)
        return self._logits(h[:, -1]), state

vocab_size = bpe_tokenizer.get_vocab_size()
torch.manual_seed(INIT_SEED)
model = LSTMLM(vocab_size, EMBED_DIM, LSTM_HIDDEN, LSTM_LAYERS).to(device)
N_PARAMS = sum(p.numel() for p in model.parameters())
print(f"LSTM language model: {N_PARAMS / 1e6:.1f}M parameters (the project range is 50M - 300M)")
assert 50e6 <= N_PARAMS <= 300e6


LSTM language model: 83.9M parameters (the project range is 50M - 300M)


## 8. Training chunks, and the held-out Supreme Court chunks used to measure progress

The new corpus is cut into `CONTEXT_LEN`-token chunks exactly as in the base run. Separately, the held-out Supreme Court documents (never trained on) are cut into chunks and a fixed random subset of them is used to measure the language-model loss before, during and after training. This is a stand-in for the hidden test set, and it is the same text for the base model and the this model, so the two numbers are comparable.

In [11]:
num_blocks = max(0, -(-(len(all_word_ids) - CONTEXT_LEN) // CONTEXT_LEN))
_flat = torch.frombuffer(all_word_ids, dtype=torch.int32) if isinstance(all_word_ids, array.array) else torch.tensor(all_word_ids, dtype=torch.int32)
train_data = _flat[:num_blocks * CONTEXT_LEN].view(num_blocks, CONTEXT_LEN).to(torch.long)
del _flat
print(f"New training chunks: {len(train_data):,}  ({len(train_data) * CONTEXT_LEN / 1e6:.0f}M tokens)")

# ---- held-out Supreme Court chunks (fixed subset, same for every model) ----
HELDOUT_EVAL_CHUNKS = 40 if QUICK_TEST else 600
_ho = []
with open(HELDOUT_FILES["supreme_court"], "r", encoding="utf-8") as f:
    for line in f:
        ids = bpe_tokenizer.encode(line.strip())
        for s in range(0, len(ids) - CONTEXT_LEN + 1, CONTEXT_LEN):
            _ho.append(ids[s:s + CONTEXT_LEN])
_sel = random.Random(1234).sample(range(len(_ho)), min(HELDOUT_EVAL_CHUNKS, len(_ho)))
heldout_chunks = torch.tensor([_ho[i] for i in sorted(_sel)], dtype=torch.long)
print(f"Held-out Supreme Court chunks for the loss check: {len(heldout_chunks)} (of {len(_ho)} available)")
del _ho


def heldout_loss(n_chunks=None):
    """Mean next-token cross-entropy (nats/token) of the current model on the held-out Supreme Court chunks, plus perplexity."""
    was_training = model.training
    model.eval()
    chunks = heldout_chunks if n_chunks is None else heldout_chunks[:n_chunks]
    total, count = 0.0, 0
    with torch.no_grad():
        for s in range(0, len(chunks), BATCH_SIZE):
            batch = chunks[s:s + BATCH_SIZE].to(device)
            logits = model(batch[:, :-1])
            total += F.cross_entropy(logits.reshape(-1, logits.shape[-1]), batch[:, 1:].reshape(-1), reduction="sum").item()
            count += batch[:, 1:].numel()
    if was_training:
        model.train()
    loss = total / max(1, count)
    return loss, math.exp(loss)


New training chunks: 179,265  (92M tokens)
Held-out Supreme Court chunks for the loss check: 600 (of 3999 available)


## 9. Training (time-limited, both GPUs via DataParallel, fp16)

The learning rate follows a cosine over whichever is further along: the fraction of `TRAIN_MINUTES` used, or the fraction of the single pass over the data done. The loop stops when either reaches 100%. The checkpoint is written (atomically) every `CKPT_EVERY_MINUTES` minutes and pushed to the bucket; if it exists when the notebook starts and is unfinished, training resumes from it.

In [12]:
n_gpus = torch.cuda.device_count() if device == "cuda" else 0
dp_model = nn.DataParallel(model) if n_gpus > 1 else model
global_batch = PER_GPU_BATCH * max(1, n_gpus)
n_chunks = len(train_data)
steps_per_epoch = n_chunks // global_batch
total_steps = steps_per_epoch * NUM_EPOCHS
assert total_steps > 0, "not enough training data for one batch"
print(f"{n_chunks:,} training chunks of {CONTEXT_LEN} tokens; {n_gpus} GPU(s) x {PER_GPU_BATCH} = global batch {global_batch}; {total_steps:,} steps = one pass; time budget {TRAIN_MINUTES} min")

use_amp = device == "cuda"
optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, betas=(0.9, 0.99), weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
step, elapsed_prev, finished = 0, 0.0, False

try_pull_checkpoint_from_hub(CHECKPOINT_PATH)
if os.path.exists(CHECKPOINT_PATH):
    ck = torch.load(CHECKPOINT_PATH, map_location=device)
    if ck.get("global_batch") == global_batch and ck.get("n_chunks") == n_chunks:
        model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
        if ck.get("scaler"): scaler.load_state_dict(ck["scaler"])
        step, elapsed_prev, finished = ck["step"], ck["elapsed"], ck["finished"]
        print(f"Resuming from step {step:,} ({elapsed_prev / 60:.1f} min of training already done; finished = {finished})")
    else:
        print("An old checkpoint with a different data/batch setup exists: ignoring it and starting over.")
    del ck

g = torch.Generator().manual_seed(DATA_SEED)
order = torch.cat([torch.randperm(n_chunks, generator=g) for _ in range(NUM_EPOCHS)])


def save_checkpoint(is_final=False):
    tmp = CHECKPOINT_PATH + ".tmp"
    torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), scaler=scaler.state_dict(), step=step,
                    elapsed=elapsed_prev + time.time() - t_start, finished=is_final, global_batch=global_batch, n_chunks=n_chunks), tmp)
    os.replace(tmp, CHECKPOINT_PATH)
    try:
        push_checkpoint_to_hub(CHECKPOINT_PATH, quiet=True); sync_run_log()
    except Exception as e:
        print(f"(push failed: {type(e).__name__})")


def lr_now(frac):
    warm = min(1.0, (step + 1) / WARMUP_STEPS)
    return PEAK_LR * warm * (MIN_LR_FRACTION + (1 - MIN_LR_FRACTION) * 0.5 * (1 + math.cos(math.pi * min(1.0, frac))))


if not finished:
    h0, p0 = heldout_loss(); print(f"held-out Supreme Court loss at the start: {h0:.3f} nats/token (ln {vocab_size} = {math.log(vocab_size):.2f})")
    model.train()
    t_start = time.time(); last_ckpt = t_start; win_loss, win_n, win_t, tok_seen = 0.0, 0, time.time(), 0
    while True:
        elapsed = elapsed_prev + time.time() - t_start
        frac = max(step / total_steps, elapsed / (TRAIN_MINUTES * 60))
        if frac >= 1.0 or step >= total_steps:
            break
        for gp in optimizer.param_groups: gp["lr"] = lr_now(frac)
        idx = order[step * global_batch:(step + 1) * global_batch]
        batch = train_data[idx].to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            loss = dp_model(batch[:, :-1], batch[:, 1:]).mean()
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        step += 1
        win_loss += loss.item(); win_n += 1; tok_seen += batch[:, 1:].numel()
        if step % 100 == 0:
            dt = time.time() - win_t
            print(f"step {step:,}/{total_steps:,}  loss {win_loss / win_n:.3f}  lr {optimizer.param_groups[0]['lr']:.2e}  {tok_seen / dt / 1000:.1f}k tok/s  done {100 * frac:.0f}%  ({elapsed / 60:.1f} min)")
            win_loss, win_n, win_t, tok_seen = 0.0, 0, time.time(), 0
        if step % 1500 == 0:
            model.eval(); h, _p = heldout_loss(100); model.train(); print(f"   held-out loss (100 chunks) {h:.3f}")
        if time.time() - last_ckpt > CKPT_EVERY_MINUTES * 60:
            save_checkpoint(); last_ckpt = time.time()
    finished = True
    save_checkpoint(is_final=True)
    print(f"Training done: {step:,} steps in {(elapsed_prev + time.time() - t_start) / 60:.1f} min")
model.eval()
final_loss, final_ppl = heldout_loss()
print(f"Final held-out Supreme Court loss: {final_loss:.3f} nats/token (perplexity {final_ppl:.1f}); the MoE models' losses are per token of the same tokenizer and chunk length 2048, so only roughly comparable")
log_event("lstm_final_heldout_loss", loss=round(final_loss, 4), steps=step)
sync_run_log()


179,265 training chunks of 512 tokens; 2 GPU(s) x 16 = global batch 32; 5,602 steps = one pass; time budget 90 min
No matching checkpoint on the Hub yet (FileNotFoundError) -- starting fresh.
held-out Supreme Court loss at the start: 10.601 nats/token (ln 40000 = 10.60)


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


step 100/5,602  loss 8.507  lr 5.00e-04  27.2k tok/s  done 2%  (1.0 min)
step 200/5,602  loss 7.939  lr 9.97e-04  24.7k tok/s  done 4%  (2.1 min)
step 300/5,602  loss 7.770  lr 1.49e-03  24.1k tok/s  done 5%  (3.2 min)
step 400/5,602  loss 7.161  lr 1.48e-03  24.2k tok/s  done 7%  (4.4 min)
step 500/5,602  loss 6.792  lr 1.47e-03  24.2k tok/s  done 9%  (5.5 min)
step 600/5,602  loss 6.501  lr 1.46e-03  24.2k tok/s  done 11%  (6.6 min)
step 700/5,602  loss 6.279  lr 1.45e-03  24.1k tok/s  done 12%  (7.7 min)
step 800/5,602  loss 6.017  lr 1.43e-03  24.2k tok/s  done 14%  (8.9 min)
step 900/5,602  loss 5.848  lr 1.42e-03  24.2k tok/s  done 16%  (10.0 min)
step 1,000/5,602  loss 5.686  lr 1.40e-03  24.1k tok/s  done 18%  (11.1 min)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

... (39 outputs left out)

/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

## 10. Generation code (LSTM state carried over; top-p sampling, no-repeat n-gram blocking)

In [13]:
PAD_ID = 0   # the BPE tokenizer's special_tokens list put "<pad>" first, so its id is 0

# ---- Decoding knobs (TEMPERATURE itself lives in the Settings cell) -----------
# Plain greedy decoding (TEMPERATURE = 0) tends to fall into loops on small models,
# so the defaults here are a starting point that fights repetition; Step 15 has a
# sweep that scores several settings on held-out Supreme Court examples so you can
# choose from numbers. For plain greedy: TEMPERATURE = 0, NO_REPEAT_NGRAM = 0.
# To mimic the official starter: TEMPERATURE = 0.1 (Settings cell), TOP_P = 0.95.
TOP_P = 0.9                # nucleus sampling, used only when TEMPERATURE > 0: sample from the
                           # smallest set of tokens whose probabilities add up to TOP_P. 1.0 = off.
REPETITION_PENALTY = 1.0   # > 1 makes tokens this continuation has ALREADY generated less likely
                           # (CTRL / Hugging Face style: positive logits divided, negative ones
                           # multiplied). Applies to generated tokens only, not the prompt, so
                           # names and terms from the prompt can still be reused. 1.0 = off.
                           # Note it hits ALL reused tokens (incl. "the", "of"), so keep it small (1.05-1.15).
NO_REPEAT_NGRAM = 4        # never let the continuation repeat any sequence of this many TOKENS
                           # (Hugging Face `no_repeat_ngram_size`, on generated tokens only). More
                           # surgical than a penalty: it only blocks exact repeats, so loops die but
                           # ordinary words can be reused. 0 = off; 4 tokens is roughly 3 words.
TOP_K = 0                  # keep only the K most likely tokens before sampling (0 = off)
MIN_P = 0.0                # drop tokens whose probability is below MIN_P x the top token's probability (0 = off)
GEN_SEED = 0               # sampling seed, set at the start of every generate_* call so a run is
                           # reproducible (only matters when TEMPERATURE > 0). None = don't seed.


def fp16_autocast():
    # T4 GPUs have dedicated fp16 tensor cores; no gradients here, so none of the
    # usual mixed-precision-training complications apply. No-op on CPU.
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda"))


class NoRepeatNgram:
    """Bans any token that would complete an n-token sequence this continuation has
    already generated (like Hugging Face's no_repeat_ngram_size, but looking only at
    the GENERATED tokens, so words and phrases can still be copied from the prompt)."""

    def __init__(self, batch_size, n):
        assert n >= 2, "NO_REPEAT_NGRAM must be 0 (off) or >= 2"
        self.n = n
        self.tokens = [[] for _ in range(batch_size)]
        self.follow = [dict() for _ in range(batch_size)]   # (n-1)-token prefix -> tokens that followed it

    def _prefix(self, toks):
        return tuple(toks[len(toks) - (self.n - 1):])

    def banned(self, vocab_size):
        """(batch, vocab_size) bool mask: True = this token would repeat an n-gram."""
        mask = torch.zeros(len(self.tokens), vocab_size, dtype=torch.bool, device=device)
        for row, toks in enumerate(self.tokens):
            if len(toks) >= self.n - 1:
                blocked = self.follow[row].get(self._prefix(toks))
                if blocked:
                    mask[row, list(blocked)] = True
        return mask

    def update(self, next_list, done):
        """Record the token each still-generating row just produced."""
        for row, token in enumerate(next_list):
            if done[row]:
                continue
            toks = self.tokens[row]
            if len(toks) >= self.n - 1:
                self.follow[row].setdefault(self._prefix(toks), set()).add(token)
            toks.append(token)


def sample_next_word_ids_batch(logits_batch, temperature, seen=None, banned=None):
    """Pick the next token for every row. logits_batch: (batch, vocab_size).
    seen: optional (batch, vocab_size) bool -- tokens each row has already
    generated (only used when REPETITION_PENALTY != 1). banned: optional
    (batch, vocab_size) bool -- tokens that may not be picked (n-gram blocking).
    Returns (batch,) ids. Order: ban n-gram repeats -> repetition penalty ->
    (temperature 0: argmax) -> temperature -> top-p filter -> sample."""
    if banned is not None:
        logits_batch = logits_batch.masked_fill(banned, float("-inf"))
    if seen is not None and REPETITION_PENALTY != 1.0:
        penalised = torch.where(logits_batch > 0, logits_batch / REPETITION_PENALTY,
                                logits_batch * REPETITION_PENALTY)
        logits_batch = torch.where(seen, penalised, logits_batch)
    if temperature == 0:
        return logits_batch.argmax(dim=-1)
    logits_batch = logits_batch / temperature
    if TOP_K > 0:
        kth = torch.topk(logits_batch, min(TOP_K, logits_batch.shape[-1]), dim=-1).values[..., -1:]
        logits_batch = logits_batch.masked_fill(logits_batch < kth, float("-inf"))
    if MIN_P > 0.0:
        _p = F.softmax(logits_batch, dim=-1)
        logits_batch = logits_batch.masked_fill(_p < MIN_P * _p.max(dim=-1, keepdim=True).values, float("-inf"))
    if TOP_P < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits_batch, descending=True, dim=-1)
        sorted_probs = F.softmax(sorted_logits, dim=-1)
        # drop every token whose predecessors already add up to more than TOP_P
        # (so the single most likely token is always kept)
        drop = (sorted_probs.cumsum(dim=-1) - sorted_probs) > TOP_P
        sorted_logits = sorted_logits.masked_fill(drop, float("-inf"))
        logits_batch = torch.full_like(logits_batch, float("-inf")).scatter(-1, sorted_idx, sorted_logits)
    probabilities = F.softmax(logits_batch, dim=-1)
    return torch.multinomial(probabilities, num_samples=1).squeeze(-1)


def _track_seen(seen, next_ids, done):
    """Record the tokens just generated for rows that were still generating."""
    if seen is None:
        return
    rows = torch.arange(len(done), device=next_ids.device)
    still_generating = torch.tensor([not d for d in done], device=next_ids.device)
    seen[rows, next_ids] = seen[rows, next_ids] | still_generating


def _pad_windows(windows):
    """Left-pad a list of token-id lists to a common length. Returns the padded
    id tensor, the per-row pad counts (list), and the pad counts as a tensor."""
    width = max(len(w) for w in windows)
    pad_lens = [width - len(w) for w in windows]
    x = torch.tensor([[PAD_ID] * pad_lens[i] + windows[i] for i in range(len(windows))],
                     dtype=torch.long, device=device)
    return x, pad_lens, torch.tensor(pad_lens, dtype=torch.long, device=device)


def _finish(results, batch_indices, generated, num_words):
    for row, original_index in enumerate(batch_indices):
        words = decode(generated[row]).split()
        words = words[:num_words]
        while len(words) < num_words:   # safety net (should basically never happen)
            words.append("<unk>")
        results[original_index] = " ".join(words)


def generate_continuations_batch(prompt_texts, num_words=128, batch_size=64, verbose=True):
    """LSTM generation: the prompt (last PROMPT_KEEP tokens) is run through the LSTM once, then one token per step with the state carried over."""
    model.eval()
    if GEN_SEED is not None:
        torch.manual_seed(GEN_SEED)
    encoded = [encode(p)[-PROMPT_KEEP:] for p in prompt_texts]
    order = sorted(range(len(prompt_texts)), key=lambda i: len(encoded[i]))   # similar lengths -> little/no left padding
    results = [None] * len(prompt_texts)
    max_new_tokens = num_words * 6   # safety cap

    for batch_start in range(0, len(order), batch_size):
        batch_indices = order[batch_start:batch_start + batch_size]
        bsz = len(batch_indices)
        generated = [[] for _ in range(bsz)]
        done = [False] * bsz
        with torch.no_grad(), fp16_autocast():
            x, pad_lens, pad_t = _pad_windows([encoded[i] for i in batch_indices])
            logits, state = model.step(x, None)
            seen = torch.zeros_like(logits, dtype=torch.bool) if REPETITION_PENALTY != 1.0 else None
            ngram = NoRepeatNgram(bsz, NO_REPEAT_NGRAM) if NO_REPEAT_NGRAM else None
            for _ in range(max_new_tokens):
                banned = ngram.banned(logits.shape[-1]) if ngram else None
                next_ids = sample_next_word_ids_batch(logits.float(), TEMPERATURE, seen, banned)
                _track_seen(seen, next_ids, done)
                next_list = next_ids.tolist()
                if ngram:
                    ngram.update(next_list, done)
                for i in range(bsz):
                    if not done[i]:
                        generated[i].append(next_list[i])
                        if len(decode(generated[i]).split()) > num_words:
                            done[i] = True
                if all(done):
                    break
                logits, state = model.step(next_ids.unsqueeze(1), state)
        _finish(results, batch_indices, generated, num_words)
        if verbose:
            print(f"  generated {min(batch_start + batch_size, len(order))} / {len(order)}")
    return results


def generate_continuation(prompt_text, num_words=128):
    """Single-prompt convenience wrapper around the batched version above."""
    return generate_continuations_batch([prompt_text], num_words=num_words,
                                         batch_size=1, verbose=False)[0]


# quick sanity check on one example before generating all of them
example_id = list(test_prompts.keys())[0]
example_continuation = generate_continuation(test_prompts[example_id])
print("Example continuation:")
print(example_continuation)
print()
print("Word count:", len(example_continuation.split()), "(should be 128)")

Example continuation:
not in dispute that the learned counsel for the petitioner has not been able to prove the case of the petitioner for the reason that the petitioner was not entitled to the relief sought for in the writ petition. The writ petition was dismissed by the learned Single Judge on the ground that the petitioner has been given the opportunity to file the writ petition before the High Court. The writ petition is accordingly dismissed. There shall be no order as to costs. J. [M.R. SHAH] J. [ABHAY MANOHAR SAPRE] J. [G.S. SINGHVI] J. [M.R. JOSEPH] J. [S.B. Sinha] J. [B.N. AGRAWAL] J. [V. K. SHAH] J. (V. K. MOHAPATRA] New Delhi, July 13, 2008 REPORTABLE IN THE SUPREME COURT OF INDIA CIVIL APPELLATE JURISDICTION CIVIL APPEAL NO.

Word count: 128 (should be 128)


## 10. Local examples and the scorer

Up to `LOCAL_EVAL_EXAMPLES` windows: one random 640-word window per held-out Supreme Court document, the first 512 words as prompt and the
next 128 as the reference. Scores are reported with a **standard error for BERTScore** (spread of per-example scores / sqrt(n)): a difference
between two settings smaller than about two standard errors should be treated as noise.

In [14]:
!pip install -q bert-score rouge-score sacrebleu
import math, random, time
from bert_score import BERTScorer
from rouge_score import rouge_scorer
import sacrebleu

EVAL_SEED = 1234
LOCAL_EVAL_EXAMPLES = 1000    # windows scored in the final re-check; the first SWEEP_EXAMPLES of them are used for the wide sweep
local_prompts, local_references = [], []
sc_path = HELDOUT_FILES.get("supreme_court")
assert sc_path, "no held-out Supreme Court file -- Section 5 did not produce one"
rng = random.Random(EVAL_SEED)
with open(sc_path, "r", encoding="utf-8") as f:
    sc_docs = [line.split() for line in f]
sc_docs = [d for d in sc_docs if len(d) >= 640]
rng.shuffle(sc_docs)
# Several non-overlapping 640-word windows per document if there are fewer than LOCAL_EVAL_EXAMPLES documents. Round-robin over the
# documents, so the first windows all come from different documents.
_starts = [[] for _ in sc_docs]
_stalled = 0
while len(local_prompts) < LOCAL_EVAL_EXAMPLES and _stalled < 3:
    _added = 0
    for di, words in enumerate(sc_docs):
        if len(local_prompts) >= LOCAL_EVAL_EXAMPLES:
            break
        for _try in range(20):
            start = rng.randint(0, len(words) - 640)
            if all(abs(start - p) >= 640 for p in _starts[di]):
                _starts[di].append(start)
                window = words[start:start + 640]
                local_prompts.append(" ".join(window[:512]))
                local_references.append(" ".join(window[512:]))
                _added += 1
                break
    _stalled = _stalled + 1 if _added == 0 else 0
print(f"Built {len(local_prompts)} local evaluation windows from {len(sc_docs)} held-out Supreme Court documents "
      f"({max(len(x) for x in _starts)} windows per document at most)")

GEN_BATCH_SIZE = 64   # lower this if you hit a CUDA out-of-memory error
_bert_scorer, _rouge = None, rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def repeated_4gram_rate(texts):
    repeated = total = 0
    for text in texts:
        words, seen = text.split(), set()
        for i in range(len(words) - 3):
            gram = tuple(words[i:i + 4])
            total += 1
            repeated += gram in seen
            seen.add(gram)
    return repeated / max(1, total)


def score_local(predictions, references):
    global _bert_scorer
    if _bert_scorer is None:
        _bert_scorer = BERTScorer(lang="en", batch_size=16)
    _, _, f1 = _bert_scorer.score(predictions, references)
    f1 = f1.float()
    rouge_l = [_rouge.score(ref, pred)["rougeL"].fmeasure for pred, ref in zip(predictions, references)]
    return {"BERTScore F1": f1.mean().item(),
            "BERT SE": (f1.std().item() / math.sqrt(len(f1))) if len(f1) > 1 else 0.0,
            "ROUGE-L": sum(rouge_l) / len(rouge_l),
            "BLEU-4": sacrebleu.corpus_bleu(predictions, [references]).score,
            "repeated 4-grams": repeated_4gram_rate(predictions),
            "_f1": f1.tolist()}


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 21.8 MB/s eta 0:00:00
Built 1000 local evaluation windows from 609 held-out Supreme Court documents (2 windows per document at most)


## 11. Pick the decoding setting on held-out Supreme Court windows

In [15]:
import zipfile
def set_knobs(temperature, top_p, ngram):
    global TEMPERATURE, TOP_P, NO_REPEAT_NGRAM, REPETITION_PENALTY, TOP_K, MIN_P
    TEMPERATURE, TOP_P, NO_REPEAT_NGRAM, REPETITION_PENALTY, TOP_K, MIN_P = temperature, top_p, ngram, 1.0, 0, 0.0

LSTM_DECODINGS = {
    "greedy_ng4":  dict(temperature=0.0, top_p=1.0, ngram=4),
    "t0.3_p0.9_ng4": dict(temperature=0.3, top_p=0.9, ngram=4),
    "t0.7_p0.9_ng4": dict(temperature=0.7, top_p=0.9, ngram=4),
}
N_LOCAL = 24 if QUICK_TEST else 200
local_results = {}
for name, k in LSTM_DECODINGS.items():
    set_knobs(**k)
    t0 = time.time()
    preds = generate_continuations_batch(local_prompts[:N_LOCAL], verbose=False)
    r = score_local(preds, local_references[:N_LOCAL])
    local_results[name] = r
    print(f"{name:16s} BERTScore F1 {r['BERTScore F1']:.4f}   ROUGE-L {r['ROUGE-L']:.4f}   BLEU-4 {r['BLEU-4']:.2f}   repeated 4-grams {r['repeated 4-grams']:.4f}   ({(time.time() - t0) / 60:.1f} min)")
    log_event("lstm_local_score", config=name, windows=N_LOCAL, **{kk: v for kk, v in r.items() if kk != "_f1"})
BEST = max(local_results, key=lambda n: local_results[n]["BERTScore F1"])
print("Best on the held-out windows:", BEST)
sync_run_log()


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (5 outputs left out)

t0.3_p0.9_ng4    BERTScore F1 0.8217   ROUGE-L 0.2169   BLEU-4 3.79   repeated 4-grams 0.0000   (0.5 min)
t0.7_p0.9_ng4    BERTScore F1 0.8245   ROUGE-L 0.2129   BLEU-4 3.81   repeated 4-grams 0.0000   (0.4 min)
Best on the held-out windows: t0.7_p0.9_ng4


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

## 12. Generate the real test continuations and write the submission

In [16]:
set_knobs(**LSTM_DECODINGS[BEST])
test_ids = list(test_prompts.keys())
if QUICK_TEST:
    test_ids = test_ids[:16]
test_texts = [test_prompts[i] for i in test_ids]
t0 = time.time()
final_texts = generate_continuations_batch(test_texts, verbose=True)
print(f"generated {len(final_texts)} continuations in {(time.time() - t0) / 60:.1f} min with {BEST}")
predictions = {doc_id: {"text": text} for doc_id, text in zip(test_ids, final_texts)}
prediction_path = "/kaggle/working/prediction.json"
with open(prediction_path, "w") as f:
    json.dump(predictions, f, indent=2)
bad = [d for d, p in predictions.items() if len(p["text"].split()) != 128]
assert not bad, f"{len(bad)} predictions do not have exactly 128 words: {bad[:5]}"
if not QUICK_TEST:
    assert set(predictions) == set(test_prompts), "prediction ids differ from the test ids"
submission_path = f"/kaggle/working/submission_lightweight_lstm84M_{BEST}{'_QUICKTEST' if QUICK_TEST else ''}.zip"
with zipfile.ZipFile(submission_path, "w") as zf:
    zf.write(prediction_path, arcname="prediction.json")     # prediction.json must be at the ROOT of the zip
print(f"All checks passed. Saved {submission_path}  (download it from the Output panel and upload it to Codabench)")
log_event("lstm_submission", config=BEST, zip=os.path.basename(submission_path), quick_test=QUICK_TEST)
sync_run_log()


  generated 64 / 1000
  generated 128 / 1000
  generated 192 / 1000
  generated 256 / 1000
  generated 320 / 1000
  generated 384 / 1000
  generated 448 / 1000
  generated 512 / 1000
  generated 576 / 1000
  generated 640 / 1000
  generated 704 / 1000
  generated 768 / 1000
  generated 832 / 1000
  generated 896 / 1000
  generated 960 / 1000
  generated 1000 / 1000
generated 1000 continuations in 0.7 min with t0.7_p0.9_ng4
All checks passed. Saved /kaggle/working/submission_lightweight_lstm84M_t0.7_p0.9_ng4.zip  (download it from the Output panel and upload it to Codabench)


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            